In [1]:
import pandas as pd
print(pd.__version__)

3.1.1


In [2]:
# Load the dataset
df = pd.read_csv("heart_disease.csv")

# How many rows and columns?
print(df.shape)

# Show the first 5 patients
df.head()

(918, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [4]:
print("Cholestrol = 0:",(df["Cholesterol"]==0).sum())
print("RestingBP = 0:",(df["RestingBP"]==0).sum())
print("Oldpeak < 0:",(df["Oldpeak"] < 0).sum())


Cholestrol = 0: 172
RestingBP = 0: 1
Oldpeak < 0: 13


In [5]:
import numpy as np
#make a copy so the original stays untouched
df_clean=df.copy()
#replace impossible zeros is NaN 
df_clean["Cholesterol"]=df_clean["Cholesterol"].replace(0,np.nan)
df_clean["RestingBP"]=df_clean["RestingBP"].replace(0,np.nan)
#count missing values in column 
df_clean.isna().sum()

Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS           0
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak             0
ST_Slope            0
HeartDisease        0
dtype: int64

## Data quality findings

- **Cholesterol = 0:** 172 patients (18.7%). Medically impossible, so treated as missing (NaN). Large and likely concentrated in one hospital, so a sensitivity analysis is needed.
- **RestingBP = 0:** 1 patient (0.1%). Treated as missing.
- **Oldpeak < 0:** 13 patients (1.4%). Possible (ST elevation) but inconsistent with the variable's definition. Kept and flagged; to be checked by hospital source.
- Original data kept unchanged in `df`; cleaned version in `df_clean`.

In [6]:
import os
import urllib.request

# Create a folder for the original raw files
os.makedirs("data/raw", exist_ok=True)

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"
files = {
    "cleveland": "processed.cleveland.data",
    "hungary": "processed.hungarian.data",
    "switzerland": "processed.switzerland.data",
    "va_longbeach": "processed.va.data",
}

for hospital, filename in files.items():
    save_path = f"data/raw/{filename}"
    urllib.request.urlretrieve(base_url + filename, save_path)
    print(f"Downloaded {hospital}: {save_path}")

Downloaded cleveland: data/raw/processed.cleveland.data
Downloaded hungary: data/raw/processed.hungarian.data
Downloaded switzerland: data/raw/processed.switzerland.data
Downloaded va_longbeach: data/raw/processed.va.data


In [7]:
# The 14 column names, in the order UCI uses
uci_columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

hospital_frames = []

for hospital, filename in files.items():
    data = pd.read_csv(
        f"data/raw/{filename}",
        header=None,          # the file has no header row
        names=uci_columns,    # so we supply the names ourselves
        na_values="?"         # UCI marks missing values with "?"
    )
    data["source"] = hospital # label every row with its hospital
    hospital_frames.append(data)
    print(f"{hospital}: {len(data)} patients")

# Stack all four hospitals into one table
uci = pd.concat(hospital_frames, ignore_index=True)
print("Total:", len(uci))

cleveland: 303 patients
hungary: 294 patients
switzerland: 123 patients
va_longbeach: 200 patients
Total: 920


In [8]:
(uci["chol"] == 0).groupby(uci["source"]).sum()

source
cleveland         0
hungary           0
switzerland     123
va_longbeach     49
Name: chol, dtype: int64

Step	Leakage risk	Rule
Imputing missing cholesterol (KNN)	The imputer "sees" HeartDisease when choosing neighbours	Impute using features only
Scaling / encoding	Fitted with label or hospital included	Fit on features only
Autoencoder training	Label used as input or to tune it	Input = features only; tune by reconstruction error only
Choosing the number of clusters (k)	Picking the k that best matches HeartDisease	Choose k using internal measures only (Silhouette, elbow, stability)
Choosing PCA components, AE size, seeds	Picking the settings that give the best ARI against the label	Decide settings before looking at the label
Evaluation	—	HeartDisease and source are used only here, after clustering is finished

In [13]:
# Translation tables: UCI number code -> Kaggle label
sex_map = {1: "M", 0: "F"}
cp_map = {1: "TA", 2: "ATA", 3: "NAP", 4: "ASY"}
restecg_map = {0: "Normal", 1: "ST", 2: "LVH"}
exang_map = {1: "Y", 0: "N"}
slope_map = {1: "Up", 2: "Flat", 3: "Down"}

# Build a new table in Kaggle's format
uci_k = pd.DataFrame({
    "Age": uci["age"],
    "Sex": uci["sex"].map(sex_map),
    "ChestPainType": uci["cp"].map(cp_map),
    "RestingBP": uci["trestbps"],
    "Cholesterol": uci["chol"],
    "FastingBS": uci["fbs"],
    "RestingECG": uci["restecg"].map(restecg_map),
    "MaxHR": uci["thalach"],
    "ExerciseAngina": uci["exang"].map(exang_map),
    "Oldpeak": uci["oldpeak"],
    "ST_Slope": uci["slope"].map(slope_map),
    "HeartDisease": (uci["num"] > 0).astype(int),
    "source": uci["source"],
})

uci_k.head()


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,source
0,63.0,M,TA,145.0,233.0,1.0,LVH,150.0,N,2.3,Down,0,cleveland
1,67.0,M,ASY,160.0,286.0,0.0,LVH,108.0,Y,1.5,Flat,1,cleveland
2,67.0,M,ASY,120.0,229.0,0.0,LVH,129.0,Y,2.6,Flat,1,cleveland
3,37.0,M,NAP,130.0,250.0,0.0,Normal,187.0,N,3.5,Down,0,cleveland
4,41.0,F,ATA,130.0,204.0,0.0,LVH,172.0,N,1.4,Up,0,cleveland


In [14]:
uci_k.isna().sum()

Age                 0
Sex                 0
ChestPainType       0
RestingBP          59
Cholesterol        30
FastingBS          90
RestingECG          2
MaxHR              55
ExerciseAngina     55
Oldpeak            62
ST_Slope          309
HeartDisease        0
source              0
dtype: int64

In [11]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [15]:
uci_k.isna().groupby(uci_k["source"]).sum()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,source
source,,,,,,,,,,,,,
cleveland,0,0,0,0,0,0,0,0,0,0,0,0,0
hungary,0,0,0,1,23,8,1,1,1,0,190,0,0
switzerland,0,0,0,2,0,75,1,1,1,6,17,0,0
va_longbeach,0,0,0,56,7,7,0,53,53,56,102,0,0


In [17]:
# Columns with no missing values: used to find candidate pairs
key_cols = ["Age", "Sex", "ChestPainType"]

# Give every patient an ID so we can track them
k = df.reset_index().rename(columns={"index": "kaggle_id"})
u = uci_k.reset_index().rename(columns={"index": "uci_id"})

# Make Age the same type in both tables (UCI has floats)
u["Age"] = u["Age"].astype(int)

# Pair every Kaggle patient with every UCI patient sharing Age, Sex and ChestPainType
pairs = k.merge(u, on=key_cols, suffixes=("_k", "_u"))

print("Candidate pairs:", len(pairs))
print("Kaggle patients with at least one candidate:", pairs["kaggle_id"].nunique())

Candidate pairs: 8060
Kaggle patients with at least one candidate: 918


In [18]:
# The remaining feature columns (not HeartDisease, not source)
other_cols = ["RestingBP", "Cholesterol", "FastingBS", "RestingECG",
              "MaxHR", "ExerciseAngina", "Oldpeak", "ST_Slope"]

# Start by assuming every pair agrees, then check column by column
agree = pd.Series(True, index=pairs.index)

for col in other_cols:
    kaggle_val = pairs[col + "_k"]
    uci_val = pairs[col + "_u"]
    same = (kaggle_val == uci_val) | uci_val.isna()
    agree = agree & same

matches = pairs[agree]

# How many UCI matches does each Kaggle patient have?
n_matches = matches.groupby("kaggle_id").size()

print("Pairs that agree on all features:", len(matches))
print("Kaggle patients with exactly 1 match:", (n_matches == 1).sum())
print("Kaggle patients with 2+ matches:", (n_matches > 1).sum())
print("Kaggle patients with no match:", 918 - len(n_matches))

Pairs that agree on all features: 927
Kaggle patients with exactly 1 match: 909
Kaggle patients with 2+ matches: 9
Kaggle patients with no match: 0


In [19]:
# The Kaggle IDs of the 9 patients with more than one match
ambiguous_ids = n_matches[n_matches > 1].index

# All their candidate matches
amb = matches[matches["kaggle_id"].isin(ambiguous_ids)]

# Show which original patient and hospital each candidate is
print(amb[["kaggle_id", "uci_id", "source"]])

# For each of the 9: how many DIFFERENT hospitals do their matches come from?
amb.groupby("kaggle_id")["source"].nunique()

      kaggle_id  uci_id        source
586          90     404       hungary
587          90     405       hungary
1788        252     488       hungary
1801        252     869  va_longbeach
2118        293     704   switzerland
2123        293     788  va_longbeach
2348        310     706   switzerland
2351        310     788  va_longbeach
3415        401     630   switzerland
3418        401     746  va_longbeach
3954        445     749  va_longbeach
3958        445     901  va_longbeach
4232        466     770  va_longbeach
4234        466     901  va_longbeach
4334        476     780  va_longbeach
4336        476     884  va_longbeach
5132        555     859  va_longbeach
5134        555     907  va_longbeach


kaggle_id
90     1
252    2
293    2
310    2
401    2
445    1
466    1
476    1
555    1
Name: source, dtype: int64

In [20]:
# Patients with exactly one match (we're sure of their hospital)
sure_ids = n_matches[n_matches == 1].index
sure = matches[matches["kaggle_id"].isin(sure_ids)]

# For each hospital: smallest Kaggle row number, largest, and how many
sure.groupby("source")["kaggle_id"].agg(["min", "max", "count"])

,min,max,count
source,,,
cleveland,615,917,303
hungary,0,292,291
switzerland,294,415,120
va_longbeach,416,614,195


In [21]:
# The "chapter" rule: row number -> hospital
def hospital_from_row(row_number):
    if row_number <= 292:
        return "hungary"
    elif row_number <= 415:
        return "switzerland"
    elif row_number <= 614:
        return "va_longbeach"
    else:
        return "cleveland"

# Add the source column to the cleaned data
df_clean["source"] = [hospital_from_row(i) for i in df_clean.index]

# How many patients per hospital?
print(df_clean["source"].value_counts())

source
cleveland       303
hungary         293
va_longbeach    199
switzerland     123
Name: count, dtype: int64


In [22]:
# For every candidate match, does the chapter rule give the same hospital?
check = matches[["kaggle_id", "source"]].copy()
check["source_by_row"] = check["kaggle_id"].apply(hospital_from_row)
check["agree"] = check["source"] == check["source_by_row"]

# For each Kaggle patient: does at least one of their matches agree with the rule?
agree_per_patient = check.groupby("kaggle_id")["agree"].any()
print("Patients where matching and row rule agree:", agree_per_patient.sum(), "of 918")

Patients where matching and row rule agree: 918 of 918


In [23]:
# Keep only matches from the correct hospital
good = matches[matches["source"] == matches["kaggle_id"].apply(hospital_from_row)]

# For each patient: does the Kaggle label equal the original label?
label_agree = (good["HeartDisease_k"] == good["HeartDisease_u"]).groupby(good["kaggle_id"]).any()
print("Patients whose label also agrees:", label_agree.sum(), "of 918")

Patients whose label also agrees: 918 of 918


In [24]:
import os
os.makedirs("data/processed", exist_ok=True)

df_clean.to_csv("data/processed/heart_with_source.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (918, 13)


## Hospital source reconstruction

- Matched 918 Kaggle patients to the 920 original UCI records (Cleveland, Hungary, Switzerland, VA Long Beach) using the 11 features only (HeartDisease excluded).
- 909 unique matches; 9 with multiple candidates, all resolved by row position (Kaggle file is stacked by hospital).
- Validation: row-position rule and feature matching agree for 918/918; HeartDisease label agrees for ___/918.
- Final counts: Hungary 293, Switzerland 123, VA 199, Cleveland 303.
- Kaggle removed 2 records: 1 Hungarian (likely duplicate: UCI 404/405), 1 VA.
- Original UCI files contain many missing values (e.g., ST_Slope missing for 309 patients, mostly Hungary and VA), but the Kaggle file has none: Kaggle values were filled in by its author. To check later.
- Saved to data/processed/heart_with_source.csv